# 第 4 讲 · 正运动学与逆运动学

贯通实验第三步：让夹爪停到方块正上方。

1. 正运动学：沿运动链把每个关节的变换连乘，算出夹爪中心，和仿真给的真值对答案
2. 逆运动学：用雅可比矩阵一步步逼近，求出让夹爪到方块上方 5 cm 的关节角
3. 发给仿真：用绝对位置模式把关节角发出去，看夹爪停在哪里

In [ ]:
# 导入：路径、XML、数值库、画图、仿真环境
from pathlib import Path   # 处理文件路径
import json   # 把结果存成 json，课件从这里读数
import xml.etree.ElementTree as ET   # 读 URDF（它是一份 XML 文件）
import numpy as np   # 矩阵运算
import matplotlib.pyplot as plt   # 画图
import gymnasium as gym   # 仿真环境的标准接口
import so101_sim   # 导入即注册星禾仿真的场景

OUT = Path("out")   # 结果存到 out/ 目录
OUT.mkdir(exist_ok=True)   # 目录不存在就新建

# 建环境并复位：种子固定为 1，方块的位置每次都一样
env = gym.make("SO101PickPlaceCube40-v1", obs_mode="rgb+depth+segmentation", control_mode="pd_joint_pos")
obs, _ = env.reset(seed=1)   # 复位，机械臂回到起始姿态
sim = env.unwrapped   # 仿真内部对象：机器人、方块都从它取
robot = sim.agent.robot   # 机器人
base = robot.links_map["base_link"]   # 基座连杆：运动学的参考坐标系


def to_base(p_world):
    """把仿真世界系里的一个点换到基座系：基座位姿的逆乘这个点。"""
    T = base.pose.to_transformation_matrix()[0].numpy()   # 基座在世界系里的 4×4 位姿
    return (np.linalg.inv(T) @ np.append(p_world, 1.0))[:3]   # 补成齐次坐标，乘逆，取前三个数

## 1 正运动学：沿运动链连乘

In [ ]:
# 读 URDF：它是一份 XML 文件，每个关节的信息都写在里面
URDF = Path(so101_sim.__file__).parent / "robots/kit_assets/kit_v1_so101.urdf"   # URDF 的位置
root = ET.parse(URDF).getroot()   # 解析 XML，取根节点
# transmission 里也有同名的 joint 标签，只取 robot 下一层的
joints = {j.get("name"): j for j in root.findall("joint")}   # 按关节名建一张表

In [ ]:
def rot_rpy(r, p, y):
    """URDF 的姿态写法 rpy：绕固定轴先 x 转 r、再 y 转 p、最后 z 转 y。"""
    cr, sr = np.cos(r), np.sin(r)   # 绕 x 轴的角：余弦、正弦
    cp, sp = np.cos(p), np.sin(p)   # 绕 y 轴的角
    cy, sy = np.cos(y), np.sin(y)   # 绕 z 轴的角
    Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])   # 绕 x 轴转 r
    Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])   # 绕 y 轴转 p
    Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])   # 绕 z 轴转 y
    return Rz @ Ry @ Rx   # 绕固定轴，后转的乘在左边


def fixed(name):
    """某个关节的固定变换：URDF 里 origin 给的平移和旋转，拼成 4×4 矩阵。"""
    o = joints[name].find("origin")   # 取这个关节的 origin 标签
    xyz = [float(v) for v in o.get("xyz").split()]   # 平移三个数，单位米
    rpy = [float(v) for v in o.get("rpy").split()]   # 姿态三个角，单位弧度
    T = np.eye(4)   # 先放一个单位矩阵
    T[:3, :3] = rot_rpy(*rpy)   # 左上 3×3 放旋转
    T[:3, 3] = xyz   # 右上 3×1 放平移
    return T

In [ ]:
ARM = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"]   # 五个臂关节


def rot_z(q):
    """绕 z 轴转 q（弧度）的 4×4 矩阵：这条运动链上每个关节的转轴都是 z。"""
    T = np.eye(4)   # 先放一个单位矩阵
    T[:2, :2] = [[np.cos(q), -np.sin(q)], [np.sin(q), np.cos(q)]]   # 左上角放二维旋转
    return T


def fk(q, g):
    """正运动学：五个臂关节角 q 与夹爪关节角 g（弧度）→ 夹爪中心在基座系里的位置。"""
    T = np.eye(4)   # 从基座出发
    for name, qi in zip(ARM, q):   # 沿运动链，一个关节一个关节往后乘
        T = T @ fixed(name) @ rot_z(qi)   # 固定部分 × 转动部分
    tip1 = T @ fixed("finger1_tip_joint")   # 定指的指尖：固定在夹爪连杆上
    tip2 = T @ fixed("gripper") @ rot_z(g) @ fixed("finger2_tip_joint")   # 动指的指尖
    return (tip1[:3, 3] + tip2[:3, 3]) / 2   # 夹爪中心 = 两个指尖的中点

In [ ]:
# 对答案：让仿真里的机械臂摆到几组关节角，比较 fk 与仿真读出的夹爪中心
poses_deg = [[-5, -102, 93, 64, -1], [20, -60, 60, 40, 0],   # 四组关节角（度）
             [-30, -30, 40, 60, 30], [45, -80, 80, 20, -20]]   # 都在限位之内
g = obs["agent"]["noisy_qpos"][0, 5].item()   # 从观测里读夹爪关节角（弧度）
rows = []   # 存每一组的结果
for deg in poses_deg:   # 一组一组地比
    q = np.deg2rad(deg)   # 度 → 弧度
    robot.set_qpos(np.append(q, g)[None, :].astype(np.float32))   # 直接把机械臂摆到这组关节角
    truth = to_base(sim.agent.tcp_pos[0].numpy())   # 仿真给的夹爪中心真值（基座系）
    mine = fk(q, g)   # 我们自己算的
    err = np.linalg.norm(mine - truth) * 1000   # 两者相差多少毫米
    rows.append((deg, mine, truth, err))   # 记下来
    print(f"fk {mine.round(4)}  真值 {truth.round(4)}  相差 {err:.4f} mm")   # 打印对照

In [ ]:
# 存下结果，课件从这个文件里读数
json.dump({"gripper_rad": g,   # 夹爪关节角
           "rows": [{"q_deg": d, "fk_m": m.tolist(), "truth_m": t.tolist(), "error_mm": float(e)}   # 每组：关节角、fk、真值、误差
                    for d, m, t, e in rows]}, open(OUT / "04-fk.json", "w"), indent=1)   # 存成文件

## 2 逆运动学：用雅可比矩阵逼近

In [ ]:
# 先读方块的位置，目标定在它正上方 5 cm
obs, _ = env.reset(seed=1)   # 复位，机械臂回到起始姿态
cube = to_base(sim.item.pose.p[0].numpy())   # 方块中心的仿真真值（基座系）
target = cube + np.array([0.0, 0.0, 0.05])   # 目标：方块中心正上方 5 cm
q0 = obs["agent"]["noisy_qpos"][0, :5].numpy().astype(float)   # 出发的关节角：五个臂关节（弧度）
g = obs["agent"]["noisy_qpos"][0, 5].item()   # 夹爪关节角，迭代中不动
print("方块", cube.round(4), "目标", target.round(4))   # 打印方块与目标
print("夹爪中心", fk(q0, g).round(4))   # 打印出发时的夹爪中心

In [ ]:
def jacobian(q, g, h=1e-6):
    """雅可比矩阵 J（3×5）：第 i 列 = 第 i 个关节多转一点点，夹爪中心怎么动。用有限差分求。"""
    J = np.zeros((3, 5))   # 先放一个 3×5 的零矩阵
    p0 = fk(q, g)   # 当前的夹爪中心
    for i in range(5):   # 一个关节一个关节地试
        dq = np.zeros(5)   # 五个关节的改变量，先全是 0
        dq[i] = h   # 只让第 i 个关节多转 h 弧度
        J[:, i] = (fk(q + dq, g) - p0) / h   # 位置的变化 ÷ 角度的变化
    return J

In [ ]:
# 关节限位也写在 URDF 里：迭代时每走一步，都把关节角夹回限位之内
lo = np.array([float(joints[n].find("limit").get("lower")) for n in ARM])   # 五个关节的下限
hi = np.array([float(joints[n].find("limit").get("upper")) for n in ARM])   # 五个关节的上限

In [ ]:
lam = 0.01   # 阻尼 λ：太小，靠近奇异位形时一步会迈得很大
q = q0.copy()   # 从起始关节角出发
history = [float(np.linalg.norm(target - fk(q, g)) * 1000)]   # 每一步的末端误差（毫米），第 0 步是出发时的
while history[-1] >= 1.0 and len(history) <= 100:   # 误差还没进 1 mm、步数没到 100，就继续
    dp = target - fk(q, g)   # 末端还差多少 Δp
    J = jacobian(q, g)   # 当前位形的雅可比矩阵
    q = q + J.T @ np.linalg.solve(J @ J.T + lam**2 * np.eye(3), dp)   # 按公式改一步 Δq
    q = np.clip(q, lo, hi)   # 夹回限位之内
    history.append(float(np.linalg.norm(target - fk(q, g)) * 1000))   # 记下这一步之后的误差
print(f"迭代 {len(history) - 1} 步，末端误差 {history[-1]:.3f} mm")   # 迭代了几步、最后差多少
print("关节角（度）", np.rad2deg(q).round(1))   # 解出来的五个关节角

In [ ]:
# 检查解在不在限位内，再画出误差一步步减小的曲线
in_limits = bool(np.all((q >= lo) & (q <= hi)))   # 每个关节都在上下限之间
print("解在限位内：", in_limits)   # 打印检查结果
plt.semilogy(history, "o-")   # 末端误差随迭代次数的变化，纵轴取对数
plt.xlabel("iteration")   # 横轴：迭代次数
plt.ylabel("error (mm)")   # 纵轴：末端误差，毫米
plt.show()   # 显示

In [ ]:
# 存下结果，课件从这个文件里读数
json.dump({"cube_m": cube.tolist(), "target_m": target.tolist(),   # 方块位置、目标位置（基座系，米）
           "q_start_deg": np.rad2deg(q0).tolist(), "q_solution_deg": np.rad2deg(q).tolist(),   # 出发与解出的关节角（度）
           "error_mm": history, "iterations": len(history) - 1,   # 每一步的误差、迭代了几步
           "lambda": lam, "in_limits": in_limits},   # 阻尼、解在不在限位内
          open(OUT / "04-ik.json", "w"), indent=1)   # 存成文件

## 3 发给仿真：绝对位置模式

In [ ]:
# pd_joint_pos 模式的动作就是 6 个绝对目标：五个臂关节加夹爪
action = np.append(q, g)[None, :].astype(np.float32)   # 五个关节角加夹爪，夹爪不动，形状 (1, 6)
for _ in range(90):   # 控制频率 30 Hz，发 90 步 = 3 秒
    obs, *_ = env.step(action)   # 每一步发同一个目标，让它走到、稳住
tcp = to_base(sim.agent.tcp_pos[0].numpy())   # 夹爪中心的仿真真值（基座系）
miss = float(np.linalg.norm(tcp - target) * 1000)   # 和目标相差多少毫米
print("夹爪中心", tcp.round(4), "目标", target.round(4))   # 打印夹爪中心与目标
print(f"相差 {miss:.2f} mm")   # 打印相差多少毫米

In [ ]:
# 看两张相机画面：顶部相机、腕部相机
top = obs["sensor_data"]["top"]["rgb"][0].cpu().numpy()   # 顶部相机的画面
wrist = obs["sensor_data"]["wrist"]["rgb"][0].cpu().numpy()   # 腕部相机的画面
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))   # 左右并排两张
for ax, img, title in zip(axes, [top, wrist], ["top camera", "wrist camera"]):   # 一张一张画
    ax.imshow(img)   # 画出这张画面
    ax.set_title(title)   # 写上相机名
    ax.axis("off")   # 不画坐标轴
plt.show()   # 显示

In [ ]:
# 存下结果，课件从这些文件里读数、取画面
plt.imsave(OUT / "04-top.png", top)   # 顶部相机画面
plt.imsave(OUT / "04-wrist.png", wrist)   # 腕部相机画面
json.dump({"tcp_m": tcp.tolist(), "target_m": target.tolist(), "miss_mm": miss, "steps": 90,   # 夹爪中心、目标、相差、步数
           "cube_after_m": to_base(sim.item.pose.p[0].numpy()).tolist()},   # 夹爪停下后方块还在原处
          open(OUT / "04-send.json", "w"), indent=1)   # 存成文件